In [1]:
import pandas as pd
import numpy as np
import warnings

# Mengabaikan warning agar output lebih bersih
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 1. Memuat Data ---
try:
    df = pd.read_csv("data-bank.csv")
    # Membersihkan nama kolom (menghilangkan spasi)
    df.columns = df.columns.str.strip()
    print("Data berhasil dimuat. Total baris:", len(df))
    print(df.head())
except FileNotFoundError:
    print("Error: File 'data-bank.csv' tidak ditemukan.")
    # Jika error, df tidak akan terdefinisi, dan blok selanjutnya akan gagal.

Data berhasil dimuat. Total baris: 6819
   Bankrupt?  ROA(C) before interest and depreciation before interest  \
0          1                                           0.370594         
1          1                                           0.464291         
2          1                                           0.426071         
3          1                                           0.399844         
4          1                                           0.465022         

   ROA(A) before interest and % after tax  \
0                                0.424389   
1                                0.538214   
2                                0.499019   
3                                0.451265   
4                                0.538432   

   ROA(B) before interest and depreciation after tax  Operating Gross Margin  \
0                                           0.405750                0.601457   
1                                           0.516730                0.610235   
2         

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Pastikan 'df' sudah ada dari blok 1
if 'df' in locals():
    print("--- 2. Mempersiapkan Data ---")
    
    target_variable = 'Bankrupt?'
    
    # Memisahkan Fitur (X) dan Target (y)
    X = df.drop(target_variable, axis=1)
    y = df[target_variable]
    
    # Simpan nama kolom untuk digunakan nanti di feature importance
    feature_names = X.columns.tolist()
    
    # Data Cleaning: Menangani Missing Values
    imputer = SimpleImputer(strategy='median')
    X_imputed = imputer.fit_transform(X)
    print("Missing values ditangani (Imputasi Median).")

    # Data Transformation: Train-Test Split (WAJIB sebelum scaling)
    # Stratify=y penting untuk data imbalance
    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, 
        test_size=0.3, 
        random_state=42, 
        stratify=y
    )
    print(f"Data dibagi: {X_train.shape[0]} latih, {X_test.shape[0]} uji.")

    # Data Transformation: Feature Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    print("Fitur data telah di-scaling (StandardScaler).\n")
else:
    print("Variabel 'df' tidak ditemukan. Jalankan blok 1 terlebih dahulu.")

--- 2. Mempersiapkan Data ---
Missing values ditangani (Imputasi Median).
Data dibagi: 4773 latih, 2046 uji.
Fitur data telah di-scaling (StandardScaler).



In [3]:
from sklearn.tree import DecisionTreeClassifier

# Pastikan data latih sudah ada dari blok 2
if 'X_train_scaled' in locals() and 'y_train' in locals():
    print("--- 3. Melatih Model Decision Tree ---")
    
    # Inisialisasi model
    # class_weight='balanced' adalah kunci untuk menangani data imbalance
    model_dt = DecisionTreeClassifier(
        class_weight='balanced', 
        random_state=42
    )
    
    # Melatih model
    model_dt.fit(X_train_scaled, y_train)
    print("Model Decision Tree (model_dt) selesai dilatih.\n")
else:
    print("Data latih tidak ditemukan. Jalankan blok 2 terlebih dahulu.")

--- 3. Melatih Model Decision Tree ---
Model Decision Tree (model_dt) selesai dilatih.



In [4]:
from sklearn.metrics import classification_report, confusion_matrix, recall_score

# Pastikan model dan data uji sudah ada
if 'model_dt' in locals() and 'X_test_scaled' in locals():
    print("--- 4. Mengevaluasi Model ---")
    
    # Memprediksi data uji
    y_pred = model_dt.predict(X_test_scaled)
    
    # Menghitung Recall untuk kelas 'Bankrupt (1)'
    recall_bankrupt = recall_score(y_test, y_pred, pos_label=1)
    
    print(f"Recall (Bankrupt=1): {recall_bankrupt:.4f}  <- Metrik Kunci")
    print("\nLaporan Klasifikasi Lengkap:")
    print(classification_report(y_test, y_pred, target_names=['Not Bankrupt (0)', 'Bankrupt (1)']))
    
    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    print(f"  (FN - False Negative): {cm[1][0]}  <- Kasus bangkrut yang terlewatkan\n")
else:
    print("Model atau data uji tidak ditemukan. Jalankan blok 2 & 3 terlebih dahulu.")

--- 4. Mengevaluasi Model ---
Recall (Bankrupt=1): 0.2576  <- Metrik Kunci

Laporan Klasifikasi Lengkap:
                  precision    recall  f1-score   support

Not Bankrupt (0)       0.98      0.98      0.98      1980
    Bankrupt (1)       0.29      0.26      0.27        66

        accuracy                           0.96      2046
       macro avg       0.63      0.62      0.62      2046
    weighted avg       0.95      0.96      0.95      2046

Confusion Matrix:
[[1938   42]
 [  49   17]]
  (FN - False Negative): 49  <- Kasus bangkrut yang terlewatkan



In [5]:
# Pastikan model dan nama fitur sudah ada
if 'model_dt' in locals() and 'feature_names' in locals():
    print("--- 5. Top 5 Variabel Paling Berpengaruh ---")
    
    # Mengambil tingkat kepentingan fitur dari model
    importances = model_dt.feature_importances_
    
    # Membuat DataFrame untuk memudahkan sorting
    feature_importance_df = pd.DataFrame({
        'Feature': feature_names, 
        'Importance': importances
    })
    
    # Mengurutkan dari yang paling penting
    top_5_features = feature_importance_df.sort_values(by='Importance', ascending=False).head(5)
    
    print(top_5_features)
    print("\n--- Proses Selesai ---")
else:
    print("Model atau 'feature_names' tidak ditemukan. Jalankan blok 2 & 3 terlebih dahulu.")

--- 5. Top 5 Variabel Paling Berpengaruh ---
                                 Feature  Importance
9   Continuous interest rate (after tax)    0.425839
36                          Debt ratio %    0.096720
34                Interest Expense Ratio    0.060733
17               Net Value Per Share (C)    0.041849
91    Degree of Financial Leverage (DFL)    0.038916

--- Proses Selesai ---


In [6]:
# --- 1. Instalasi Library SMOTE ---
try:
    import imblearn
    print(f"Library 'imbalanced-learn' (imblearn) sudah terinstal. Versi: {imblearn.__version__}")
except ImportError:
    print("Library 'imblearn' belum terinstal. Mencoba menginstal...")
    # Menggunakan !pip untuk menginstal di lingkungan notebook
    !pip install imbalanced-learn
    print("Instalasi selesai. Anda mungkin perlu me-restart kernel jika ini pertama kalinya.")

Library 'imbalanced-learn' (imblearn) sudah terinstal. Versi: 0.14.0


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Pastikan 'df' sudah ada dari blok 1
if 'df' in locals():
    print("--- 2. Mempersiapkan Data (Untuk SMOTE) ---")
    
    target_variable = 'Bankrupt?'
    
    X = df.drop(target_variable, axis=1)
    y = df[target_variable]
    
    feature_names = X.columns.tolist()
    
    # 1. Imputasi Missing Values
    imputer = SimpleImputer(strategy='median')
    X_imputed = imputer.fit_transform(X)
    print("Missing values ditangani.")

    # 2. Train-Test Split (SEBELUM SMOTE)
    # Kita split data mentah yang sudah di-imputasi
    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, 
        test_size=0.3, 
        random_state=42, 
        stratify=y
    )
    print(f"Data dibagi: {X_train.shape[0]} latih, {X_test.shape[0]} uji.")
    print(f"Distribusi 'Bankrupt' di y_train (sebelum SMOTE):\n{y_train.value_counts(normalize=True)}")
else:
    print("Variabel 'df' tidak ditemukan. Jalankan blok 1 (Memuat Data) terlebih dahulu.")

--- 2. Mempersiapkan Data (Untuk SMOTE) ---
Missing values ditangani.
Data dibagi: 4773 latih, 2046 uji.
Distribusi 'Bankrupt' di y_train (sebelum SMOTE):
Bankrupt?
0    0.967735
1    0.032265
Name: proportion, dtype: float64


In [8]:
from imblearn.over_sampling import SMOTE

# Pastikan data latih sudah ada dari blok 2
if 'X_train' in locals():
    print("--- 3. Menerapkan SMOTE pada Data Latih ---")
    
    smote = SMOTE(random_state=42)
    
    # Terapkan SMOTE
    # Ini menciptakan data baru (sintetis) untuk kelas minoritas
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    
    print("SMOTE selesai diterapkan.")
    print(f"Ukuran X_train sebelum SMOTE: {X_train.shape}")
    print(f"Ukuran X_train setelah SMOTE: {X_train_res.shape}")
    print(f"\nDistribusi 'Bankrupt' di y_train (setelah SMOTE):\n{y_train_res.value_counts(normalize=True)}")
else:
    print("Data latih tidak ditemukan. Jalankan blok 2 terlebih dahulu.")

--- 3. Menerapkan SMOTE pada Data Latih ---
SMOTE selesai diterapkan.
Ukuran X_train sebelum SMOTE: (4773, 95)
Ukuran X_train setelah SMOTE: (9238, 95)

Distribusi 'Bankrupt' di y_train (setelah SMOTE):
Bankrupt?
0    0.5
1    0.5
Name: proportion, dtype: float64


In [9]:
from sklearn.preprocessing import StandardScaler

# Pastikan data dari blok 2 & 3 sudah ada
if 'X_train_res' in locals():
    print("--- 4. Scaling Data ---")
    
    scaler = StandardScaler()
    
    # FIT dan TRANSFORM pada data latih yang sudah di-SMOTE
    X_train_scaled = scaler.fit_transform(X_train_res)
    
    # HANYA TRANSFORM pada data uji (menggunakan scaler yang sama)
    X_test_scaled = scaler.transform(X_test)
    
    print("Scaling selesai. Scaler di-fit pada data latih hasil SMOTE.")
    print("Data latih dan data uji telah di-scale.\n")
else:
    print("Data hasil SMOTE tidak ditemukan. Jalankan blok 3 terlebih dahulu.")

--- 4. Scaling Data ---
Scaling selesai. Scaler di-fit pada data latih hasil SMOTE.
Data latih dan data uji telah di-scale.



In [10]:
from sklearn.tree import DecisionTreeClassifier

# Pastikan data latih yang sudah di-SMOTE dan di-scale ada
if 'X_train_scaled' in locals() and 'y_train_res' in locals():
    print("--- 5. Melatih Model (Setelah SMOTE) ---")
    
    # PENTING: Hapus class_weight='balanced'
    model_dt_smote = DecisionTreeClassifier(
        random_state=42
    )
    
    # Latih model pada data yang sudah di-resample (SMOTE)
    model_dt_smote.fit(X_train_scaled, y_train_res)
    
    print("Model Decision Tree (model_dt_smote) selesai dilatih pada data SMOTE.\n")
else:
    print("Data latih hasil SMOTE & scaling tidak ditemukan. Jalankan blok 4 terlebih dahulu.")

--- 5. Melatih Model (Setelah SMOTE) ---
Model Decision Tree (model_dt_smote) selesai dilatih pada data SMOTE.



In [11]:
from sklearn.metrics import classification_report, confusion_matrix, recall_score

# Pastikan model baru dan data uji (asli) sudah ada
if 'model_dt_smote' in locals() and 'X_test_scaled' in locals():
    print("--- 6. Mengevaluasi Model (Versi SMOTE) ---")
    
    # Prediksi pada data uji yang ASLI (tidak di-SMOTE)
    y_pred_smote = model_dt_smote.predict(X_test_scaled)
    
    recall_bankrupt_smote = recall_score(y_test, y_pred_smote, pos_label=1)
    
    print(f"Recall (Bankrupt=1): {recall_bankrupt_smote:.4f}  <- Metrik Kunci")
    print("\nLaporan Klasifikasi Lengkap:")
    print(classification_report(y_test, y_pred_smote, target_names=['Not Bankrupt (0)', 'Bankrupt (1)']))
    
    print("Confusion Matrix:")
    cm_smote = confusion_matrix(y_test, y_pred_smote)
    print(cm_smote)
    print(f"  (FN - False Negative): {cm_smote[1][0]}  <- Kasus bangkrut yang terlewatkan\n")
else:
    print("Model 'model_dt_smote' atau data uji tidak ditemukan. Jalankan blok 4 & 5 terlebih dahulu.")

--- 6. Mengevaluasi Model (Versi SMOTE) ---
Recall (Bankrupt=1): 0.4091  <- Metrik Kunci

Laporan Klasifikasi Lengkap:
                  precision    recall  f1-score   support

Not Bankrupt (0)       0.98      0.95      0.97      1980
    Bankrupt (1)       0.23      0.41      0.29        66

        accuracy                           0.94      2046
       macro avg       0.60      0.68      0.63      2046
    weighted avg       0.96      0.94      0.94      2046

Confusion Matrix:
[[1888   92]
 [  39   27]]
  (FN - False Negative): 39  <- Kasus bangkrut yang terlewatkan

